<a href="https://colab.research.google.com/github/AMLU-ANNA-JOSHY/wav2vec-2.0-SCR/blob/main/MTL_w2v_SCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Wav2vec 2.0-based SCR on 19 commands of UA-Speech**

In [ ]:
import os, json
from collections import Counter
import wave
import contextlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio
from torch.nn.functional import softmax
import pandas as pd
from collections import Counter
from transformers import Wav2Vec2Model, Wav2Vec2ForSequenceClassification
from transformers import EarlyStoppingCallback, Wav2Vec2Processor
from sklearn.metrics import accuracy_score, classification_report
from datasets import load_dataset, DatasetDict
from dataclasses import dataclass
from typing import Dict, List, Union
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/UAS_19_W2V.zip"

In [ ]:
# Mapping
severity_to_speaker = {
    "VERY_LOW": ["F05", "M08", "M09", "M10", "M14"],
    "LOW": ["F04", "M05", "M11"],
    "MEDIUM": ["F02", "M07", "M16"],
    "HIGH": ["F03", "M01", "M04", "M12"]
}

# Reverse mapping: speaker -> severity
speaker_to_severity = {spk: sev for sev, spks in severity_to_speaker.items() for spk in spks}

# **Dataset preprocessing**

- UAS_manifests_labeled contains the files holding metadata about the audio samples in the training, testing, and validation splits.
- It has separate csv files for training, testing, and validation splits with audio path, speaker info, from which severity can be derived, word code of each audio file in the UA-Speech dataset.
- The transcriptions of the audio files are made available as a codebook in the UA-Speech dataset; from which the word names are also made available.

In [ ]:
num_word_labels = 19
num_severity_labels = 4

In [ ]:
data_files = {
    "train": "/content/drive/MyDrive/uas_manifests_labeled/train_labeled.csv",
    "validation": "/content/drive/MyDrive/uas_manifests_labeled/eval_labeled.csv",
    "test": "/content/drive/MyDrive/uas_manifests_labeled/test_labeled.csv",
}

# Load CSVs as HuggingFace Dataset
dataset = load_dataset("csv", data_files=data_files)
print(dataset)

# Load pretrained Wav2Vec2 processor
model_ckpt = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(model_ckpt)

In [ ]:
# Preprocessing function
def preprocess_function(batch):
    # get audio info from file path
    speech_array, sampling_rate = torchaudio.load(batch["path"])
    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
        speech_array = resampler(speech_array)
    speech_array = speech_array.squeeze().numpy()
    inputs = processor(speech_array, sampling_rate=16000, return_tensors="pt", padding=True)
    batch["input_values"] = inputs.input_values[0]  # remove batch dimension
    return batch

# Apply preprocessing
dataset = dataset.map(preprocess_function)
print(dataset)

In [ ]:
# Rename label column for all splits
dataset = dataset.rename_column("label", "labels")
dataset.set_format(
    type="torch",
    columns=["input_values", "labels", "speaker", "severity", "word_code", "word"]
)

In [ ]:
dataset["train"][:2]

In [ ]:
from datasets import ClassLabel

# Define severity mapping
severity_classes = ['VERY_LOW', 'LOW', 'MEDIUM', 'HIGH']
severity_labels = ClassLabel(names=severity_classes)

# Encode for each split
def encode_severity(batch):
    batch["severity_labels"] = severity_labels.str2int(batch["severity"])
    return batch

dataset = dataset.map(encode_severity)

In [ ]:
dataset["train"][:2]

In [ ]:
from torch.utils.data import Dataset

class HFDatasetTorch(Dataset):
    def __init__(self, hf_dataset_split):
        self.ds = hf_dataset_split

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        input_tensor = torch.tensor(item["input_values"], dtype=torch.float32)

        return {
            "input_values": input_tensor,
            "label": int(item["labels"]),
            "severity": int(item["severity_labels"]),
            "speaker":item["speaker"]
        }

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    # batch: list of dicts
    inputs = [b["input_values"] for b in batch]
    # do padding to longest length along dim 0
    padded = pad_sequence(inputs, batch_first=True)  # [B, T_max]
    # lengths before padding
    lengths = torch.tensor([t.shape[0] for t in inputs], dtype=torch.long)
    # attention mask- 1: real audio; 0: padded zeros: don't attend to these noise
    attention_mask = (torch.arange(padded.size(1))[None, :].to(lengths.device) < lengths[:, None]).long()

    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    sev_labels = torch.tensor([b["severity"] for b in batch], dtype=torch.long)

    return {
      "input_values": padded,
      "attention_mask": attention_mask,
      "labels": labels,
      "severity_labels": sev_labels,
    }


In [ ]:
class Wav2Vec2MultiTask(nn.Module):
    def __init__(self, pretrained_ckpt="facebook/wav2vec2-base", num_words=19, \
                 num_severity=4, dropout=0.2, freeze_feature_extractor=True):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained(pretrained_ckpt)
        if freeze_feature_extractor:
            for param in self.wav2vec.feature_extractor.parameters():
                param.requires_grad = False

        hidden_size = self.wav2vec.config.hidden_size

        # word classification head
        self.word_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size//2, num_words)
        )

        # severity classification head
        self.sev_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size//2, num_severity)
        )


    def forward(self, input_values, attention_mask=None):
        """
        input_values: Tensor [batch, seq_len]
        returns dict with logits for both tasks,
        together with pooled embedding for visulaization
        """
        outputs = self.wav2vec(input_values, attention_mask=attention_mask, output_hidden_states=False, return_dict=True)
        # outputs.last_hidden_state: [batch, seq_len, hidden]
        last_hidden = outputs.last_hidden_state
        # pooling: mean over time dimension: [batch, hidden]
        pooled = last_hidden.mean(dim=1)

        word_logits = self.word_head(pooled)   # [batch, num_words]
        sev_logits  = self.sev_head(pooled)    # [batch, num_severity]

        return {
         "word_logits": word_logits,
         "sev_logits": sev_logits,
         "pooled": pooled
        }


In [ ]:
def train_multitask(model, train_loader, val_loader, device,
                    num_epochs=40, lr=2e-5, alpha=0,  # alpha: weight for severity loss
                    weight_decay=0.01, grad_clip=1.0, out_dir="./mtl_ckpt"):
    model.to(device)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=weight_decay)
    total_steps = num_epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_steps), eta_min=1e-7)

    criterion = nn.CrossEntropyLoss()
    criterion_phoneme = nn.BCEWithLogitsLoss()
    best_val_acc = -1.0
    epochs_no_improve = 0
    early_stop_patience = 5

    os.makedirs(out_dir, exist_ok=True)

    for epoch in range(1, num_epochs+1):
        print("Training: ",epoch)
        model.train()
        train_losses = []
        train_word_preds = []
        train_word_trues = []

        for batch in train_loader:

            input_values = batch["input_values"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            sev_labels = batch["severity_labels"].to(device)

            out = model(input_values, attention_mask=attention_mask)

            word_logits = out["word_logits"]
            sev_logits  = out["sev_logits"]
            loss_word = criterion(word_logits, labels)
            loss_sev  = criterion(sev_logits, sev_labels)

            loss = loss_word  + alpha * loss_sev

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            train_losses.append(loss.item())

            preds = torch.argmax(word_logits, dim=-1).cpu().numpy().tolist()
            train_word_preds.extend(preds)
            train_word_trues.extend(labels.cpu().numpy().tolist())

        scheduler.step()

        train_loss = np.mean(train_losses)
        train_acc = accuracy_score(train_word_trues, train_word_preds)

        # validation
        model.eval()
        val_losses = []
        val_word_preds = []
        val_word_trues = []
        val_sev_preds = []
        val_sev_trues = []

        with torch.no_grad():
            for batch in val_loader:

                input_values = batch["input_values"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                out = model(input_values, attention_mask=attention_mask)

                labels = batch["labels"].to(device)
                sev_labels = batch["severity_labels"].to(device)

                word_logits = out["word_logits"]
                sev_logits  = out["sev_logits"]
                loss_word = criterion(word_logits, labels)
                loss_sev  = criterion(sev_logits, sev_labels)
                loss = loss_word  + alpha * loss_sev

                val_losses.append(loss.item())

                val_word_preds.extend(torch.argmax(word_logits, dim=-1).cpu().numpy().tolist())
                val_word_trues.extend(labels.cpu().numpy().tolist())

                val_sev_preds.extend(torch.argmax(sev_logits, dim=-1).cpu().numpy().tolist())
                val_sev_trues.extend(sev_labels.cpu().numpy().tolist())

        val_loss = np.mean(val_losses)
        val_acc = accuracy_score(val_word_trues, val_word_preds)
        val_sev_acc = accuracy_score(val_sev_trues, val_sev_preds)

        print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} train_acc {train_acc:.4f} | val_loss {val_loss:.4f} val_acc {val_acc:.4f} ") # sev_acc {val_sev_acc:.4f}

        # early stopping logic based on val_acc
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stop_patience:
                print("Early stopping triggered.")
                break

    # load best model
    model.load_state_dict(torch.load(os.path.join(out_dir, "best_model.pt"), map_location=device))
    return model


# **SID EXPERIMENTS**

In [ ]:
from datasets import concatenate_datasets

full_dataset = concatenate_datasets([
    dataset["train"],
    dataset["validation"],
    dataset["test"]
])

In [ ]:
def get_data(test_speaker):

  loso_test_hf = full_dataset.filter(
    lambda x: x["speaker"] == test_speaker
  )

  train_from_train = dataset["train"].filter(
    lambda x: x["speaker"] != test_speaker
  )

  train_from_test = dataset["test"].filter(
    lambda x: x["speaker"] != test_speaker
  )

  loso_train_hf = concatenate_datasets([
    train_from_train,
    train_from_test
  ])

  loso_val_hf = dataset["validation"].filter(
    lambda x: x["speaker"] != test_speaker
  )

  return loso_train_hf, loso_val_hf, loso_test_hf

In [ ]:
def get_result(model, test_speaker, loso_test_hf):
  model.eval()
  all_word_preds, all_word_labels = [], []
  all_sev_preds, all_sev_labels = [], []

  for example in loso_test_hf:
    inputs = processor(example["input_values"], sampling_rate=16000,
                       return_tensors="pt", padding=True).input_values.to(device)
    word_label = example["labels"]

    with torch.no_grad():
        outputs = model(inputs)
        word_logits = outputs["word_logits"]

    word_pred = torch.argmax(word_logits, dim=-1).item()
    all_word_preds.append(word_pred)
    all_word_labels.append(word_label)

    return accuracy_score(all_word_labels, all_word_preds)


In [ ]:
severity_classes = ['VERY_LOW', 'LOW', 'MEDIUM', 'HIGH']

severity_to_speaker = {
    "VERY_LOW": ["F05", "M08", "M09", "M10", "M14"],
    "LOW": ["F04", "M05", "M11"],
    "MEDIUM": ["F02", "M07", "M16"],
    "HIGH": ["F03", "M01", "M04", "M12"]
}

speakers = [j for i in severity_to_speaker.values() for j in i]

In [ ]:
spk_list = []
accc = []
early_stop_patience = 5

for test_speaker in ["F03", "M01", "M04", "M12"]:
    spk_list.append(test_speaker)
    loso_train_hf, loso_val_hf, loso_test_hf = get_data(test_speaker)

    train_torch_ds = HFDatasetTorch(loso_train_hf)
    val_torch_ds   = HFDatasetTorch(loso_val_hf)
    test_torch_ds  = HFDatasetTorch(loso_test_hf)

    train_loader = DataLoader(train_torch_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_torch_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_torch_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

    print(f"Training for SID test speaker: {test_speaker}")
    print("*"*50)


    model = Wav2Vec2MultiTask("facebook/wav2vec2-base", num_word_labels, num_severity_labels) # num_phoneme_labels removed as it was deleted

    model = train_multitask(model, train_loader, val_loader, device,
                    num_epochs=2, lr=2e-5, alpha= 0.5,
                    weight_decay=0.01, grad_clip=1.0, out_dir=f"/content/drive/MyDrive/wav2vec_MLT_severity_SID_{test_speaker}_classifier")

    acc = get_result(model, test_speaker, loso_test_hf)
    print("Word Recognition Accuracy:", acc)
    accc.append(acc)
    print("="*60)
    print(spk_list, accc)
    print("="*60)

# **SD EXPERIMENTS**

# **Model definition for MTL**



In [ ]:

model = Wav2Vec2MultiTask("facebook/wav2vec2-base", num_word_labels, num_severity_labels)


In [ ]:
train_torch_ds = HFDatasetTorch(dataset["train"])
val_torch_ds   = HFDatasetTorch(dataset["validation"])
test_torch_ds  = HFDatasetTorch(dataset["test"])

train_loader = DataLoader(train_torch_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_torch_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_torch_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)


In [ ]:
early_stop_patience = 10
model = train_multitask(model, train_loader, val_loader, device,
                    num_epochs=40, lr=2e-5, alpha=0.7,
                    weight_decay=0.01, grad_clip=1.0, out_dir="/content/drive/MyDrive/wav2vec_MLT_0.7_classifier")

**TESTING**

In [ ]:
from torch.nn.functional import softmax

all_word_preds, all_word_labels = [], []
all_sev_preds, all_sev_labels = [], []

for example in dataset["validation"]:
    inputs = processor(example["input_values"], sampling_rate=16000,
                       return_tensors="pt", padding=True).input_values.to(device)
    word_label = example["labels"]
    severity_label = example["severity_labels"]

    with torch.no_grad():
        outputs = model(inputs)
        word_logits = outputs["word_logits"]
        severity_logits = outputs["sev_logits"]
        pooled = outputs["pooled"]

    word_pred = torch.argmax(word_logits, dim=-1).item()
    sev_pred = torch.argmax(severity_logits, dim=-1).item()

    all_word_preds.append(word_pred)
    all_word_labels.append(word_label)
    all_sev_preds.append(sev_pred)
    all_sev_labels.append(severity_label)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Word Recognition Accuracy:", accuracy_score(all_word_labels, all_word_preds))
print("Severity Classification Accuracy:", accuracy_score(all_sev_labels, all_sev_preds))

print("\nWord Classification Report:\n", classification_report(all_word_labels, all_word_preds))
print("\nSeverity Classification Report:\n", classification_report(all_sev_labels, all_sev_preds))


**Testing using saved checkpoint**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Wav2Vec2MultiTask(pretrained_ckpt="facebook/wav2vec2-base",
                          num_words=19,
                          num_severity=4)
model.load_state_dict(torch.load("/content/drive/MyDrive/wav2vec_MLT_0.5_classifier/best_model.pt"))
model.to(device)
model.eval()

In [ ]:
from torch.nn.functional import softmax

all_word_preds, all_word_labels = [], []
all_sev_preds, all_sev_labels = [], []

for example in dataset["test"]:
    inputs = processor(example["input_values"], sampling_rate=16000,
                       return_tensors="pt", padding=True).input_values.to(device)
    word_label = example["labels"]
    severity_label = example["severity_labels"]

    with torch.no_grad():
        outputs = model(inputs)
        word_logits = outputs["word_logits"]
        severity_logits = outputs["sev_logits"]
        pooled = outputs["pooled"]

    word_pred = torch.argmax(word_logits, dim=-1).item()
    sev_pred = torch.argmax(severity_logits, dim=-1).item()

    all_word_preds.append(word_pred)
    all_word_labels.append(word_label)
    all_sev_preds.append(sev_pred)
    all_sev_labels.append(severity_label)

In [ ]:
print("Word Recognition Accuracy:", accuracy_score(all_word_labels, all_word_preds))
print("Severity Classification Accuracy:", accuracy_score(all_sev_labels, all_sev_preds))

print("\nWord Classification Report:\n", classification_report(all_word_labels, all_word_preds))
print("\nSeverity Classification Report:\n", classification_report(all_sev_labels, all_sev_preds))
